# Weather Disruption Index — EDA Notebook

This notebook loads the cleaned hourly weather features and performs exploratory data analysis (EDA):

- Dataset overview & schema check
- Missingness summary & visualization
- Distributions and seasonal patterns
- Correlations (numerical features)
- Airport-level summaries
- Outlier checks
- Optional: save a trimmed, model-ready dataset

> **Tip:** Run the setup cell below first to install any missing packages.


In [ ]:
# --- Setup (run once if needed) ---
# This cell installs optional utilities used in this notebook.
import sys, subprocess
def pip_install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for p in ["pandas", "numpy", "matplotlib", "pyarrow"]:
    pip_install(p)

# missingno is optional for the matrix plot; we guard its import later
try:
    __import__("missingno")
except ImportError:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'missingno'])
    except Exception:
        pass  # continue without it


In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


In [ ]:
# --- Load Data (tries both common project layouts) ---
ROOT = Path.cwd()
candidates = [
    ROOT / 'cleaned' / 'hourly_features.csv',              # current layout
    ROOT / 'data' / 'processed' / 'hourly_features.csv',   # clean layout
    ROOT.parent / 'cleaned' / 'hourly_features.csv',       # if notebook is in a subfolder
    ROOT.parent / 'data' / 'processed' / 'hourly_features.csv',
]
csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find 'hourly_features.csv' in cleaned/ or data/processed/ (tried multiple locations).")

print(f"Using file: {csv_path}")
df = pd.read_csv(csv_path, parse_dates=['DATE'])
print(df.shape)
df.head()


## 1) Overview & Schema

In [ ]:
print("Columns:\n", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
display(df.describe(include='all'))
df.sample(min(5, len(df)))


## 2) Missingness Summary & (Optional) Matrix Plot

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
display(missing.to_frame('missing_frac').style.format({"missing_frac": "{:.2%}"}))

# Optional matrix plot if missingno is available
try:
    import missingno as msno
    sample = df.sample(min(len(df), 50000), random_state=42)
    plt.figure()
    msno.matrix(sample)
    plt.title('Missingness Matrix (sample)')
    plt.show()
except Exception as e:
    print("missingno not installed or failed to import; skipping matrix plot.")


## 3) Distributions (numeric)

In [ ]:
numeric_cols = ['tmp_c','dew_c','dew_spread_c','slp_hpa','vis_m','cig_m','wnd_dir','wnd_spd']
for col in numeric_cols:
    if col in df.columns:
        plt.figure()
        df[col].plot.hist(bins=60)
        plt.xlabel(col)
        plt.ylabel('count')
        plt.title(f'Distribution of {col}')
        plt.show()


## 4) Seasonal Patterns

In [ ]:
if {'month','ICAO','tmp_c'}.issubset(df.columns):
    monthly_tmp = df.groupby(['ICAO','month'])['tmp_c'].mean().unstack(0)
    plt.figure()
    monthly_tmp.plot()
    plt.title('Monthly Avg Temperature by ICAO')
    plt.xlabel('Month')
    plt.ylabel('Temperature (°C)')
    plt.legend(title='ICAO', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

if {'month','ICAO','vis_m'}.issubset(df.columns):
    monthly_vis = df.groupby(['ICAO','month'])['vis_m'].median().unstack(0)
    plt.figure()
    monthly_vis.plot()
    plt.title('Monthly Median Visibility by ICAO')
    plt.xlabel('Month')
    plt.ylabel('Visibility (m)')
    plt.legend(title='ICAO', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


## 5) Correlations (numeric only)

In [ ]:
num_present = [c for c in numeric_cols if c in df.columns]
if len(num_present) >= 2:
    corr = df[num_present].corr()
    plt.figure(figsize=(6,5))
    im = plt.imshow(corr, aspect='auto')
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(len(num_present)), num_present, rotation=45, ha='right')
    plt.yticks(range(len(num_present)), num_present)
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()
    corr
else:
    print('Not enough numeric columns to compute correlations.')


## 6) Airport-Level Summaries

In [ ]:
summary_cols = ['tmp_c','dew_c','slp_hpa','vis_m','cig_m','wnd_spd']
avail = [c for c in summary_cols if c in df.columns]
if 'ICAO' in df.columns and avail:
    agg = df.groupby('ICAO')[avail].agg(['mean','median','std','min','max']).round(2)
    agg
else:
    print('Missing ICAO or summary columns.')


## 7) Outlier Checks (simple)

In [ ]:
def iqr_outlier_mask(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return (s < (q1 - k*iqr)) | (s > (q3 + k*iqr))

for col in ['tmp_c','slp_hpa','vis_m','wnd_spd']:
    if col in df.columns:
        mask = iqr_outlier_mask(df[col].dropna())
        rate = mask.mean() if hasattr(mask, 'mean') else float(mask.sum())/len(mask)
        print(f"Outlier rate for {col}: {rate:.2%}")


## 8) Save a Trimmed, Model-Ready Subset (optional)

In [ ]:
model_cols = ['ICAO','DATE','tmp_c','dew_c','slp_hpa','vis_m','cig_m','wnd_dir','wnd_spd']
present = [c for c in model_cols if c in df.columns]
df_model = df[present].dropna()
out_dir = csv_path.parent  # same folder as the CSV
out_path = out_dir / 'hourly_features_modelready.parquet'
df_model.to_parquet(out_path, index=False)
print(f"Saved model-ready file: {out_path}  (rows={len(df_model):,})")
